### Checkagem Despesas

In [12]:
import duckdb

# Caminho base relativo ao notebook parquet_teste.ipynb
PATH = "parquet/despesas"

print("==================================================")
print("      INICIANDO VERIFICAÇÃO DAS TABELAS           ")
print("==================================================\n")

# -------------------------------------------------------------------------
# 1. TESTE DA TABELA FATO: Contagem de Linhas e Totais Financeiros
# -------------------------------------------------------------------------
print("1. CHECAGEM DA TABELA FATO (fato_empenhos)")
fato_stats = duckdb.sql(f"""
    SELECT 
        COUNT(*) AS total_linhas,
        SUM(valor_empenhado) AS total_empenhado,
        SUM(valor_liquidado) AS total_liquidado,
        SUM(valor_pago) AS total_pago
    FROM '{PATH}/fato_empenhos.parquet'
""").df()

print(f"   • Total de Linhas: {fato_stats['total_linhas'][0]:,}")
print(f"   • Total Empenhado: R$ {fato_stats['total_empenhado'][0]:,.2f}")
print(f"   • Total Liquidado: R$ {fato_stats['total_liquidado'][0]:,.2f}")
print(f"   • Total Pago:      R$ {fato_stats['total_pago'][0]:,.2f}\n")

# -------------------------------------------------------------------------
# 2. UNICIDADE DE CHAVES PRIMÁRIAS NAS DIMENSÕES (Garante 3FN sem duplicatas)
# -------------------------------------------------------------------------
print("2. CHECAGEM DE CHAVES PRIMÁRIAS DUPLICADAS NAS DIMENSÕES")

dimensoes_pks = [
    ("dim_credor", "cpf_cnpj"),
    ("dim_unidade_gestora", "codigo_unidade_gestora"),
    ("dim_unidade_orcamentaria", "codigo_unidade_orcamentaria"),
    ("dim_funcao", "codigo_funcao"),
    ("dim_subfuncao", "codigo_subfuncao"),
    ("dim_programa", "codigo_programa"),
    ("dim_acao", "codigo_acao"),
    ("dim_categoria_economica", "codigo_categoria_economica"),
    ("dim_natureza", "codigo_natureza"),
    ("dim_modalidade_aplicacao", "codigo_modalidade_aplicacao"),
    ("dim_elemento_despesa", "codigo_elemento_despesa"),
    ("dim_fonte_recurso", "codigo_fonte_recurso"),
    ("dim_co", "co")
]

for tabela, pk in dimensoes_pks:
    duplicados = duckdb.sql(f"""
        SELECT {pk}, COUNT(*) as qtd
        FROM '{PATH}/{tabela}.parquet'
        GROUP BY {pk}
        HAVING COUNT(*) > 1
    """).df()
    
    status = "OK (100% Única)" if len(duplicados) == 0 else f"ATENÇÃO: {len(duplicados)} chaves duplicadas!"
    print(f"   • {tabela} -> PK [{pk}]: {status}")

# -------------------------------------------------------------------------
# 3. INTEGRIDADE REFERENCIAL (Chaves Órfãs na Tabela Fato)
# -------------------------------------------------------------------------
print("\n3. CHECAGEM DE INTEGRIDADE REFERENCIAL (FK -> PK)")

# Verifica se existe algum credor na fato que NÃO existe na dim_credor
orfaos_credor = duckdb.sql(f"""
    SELECT COUNT(*) AS qtd
    FROM '{PATH}/fato_empenhos.parquet' f
    LEFT JOIN '{PATH}/dim_credor.parquet' c ON f.cpf_cnpj = c.cpf_cnpj
    WHERE f.cpf_cnpj IS NOT NULL AND c.cpf_cnpj IS NULL
""").df()['qtd'][0]

# Verifica se existe alguma UG na fato que NÃO existe na dim_unidade_gestora
orfaos_ug = duckdb.sql(f"""
    SELECT COUNT(*) AS qtd
    FROM '{PATH}/fato_empenhos.parquet' f
    LEFT JOIN '{PATH}/dim_unidade_gestora.parquet' u ON f.codigo_unidade_gestora = u.codigo_unidade_gestora
    WHERE f.codigo_unidade_gestora IS NOT NULL AND u.codigo_unidade_gestora IS NULL
""").df()['qtd'][0]

print(f"   • Registros na Fato com Credor inexistente na Dimensão: {orfaos_credor} (Esperado: 0)")
print(f"   • Registros na Fato com Unidade Gestora inexistente:   {orfaos_ug} (Esperado: 0)")

# -------------------------------------------------------------------------
# 4. TESTE DE CONSULTA REAL (JOIN Fato x Dimensões)
# -------------------------------------------------------------------------
print("\n4. TESTE PRÁTICO DE JOIN (Top 5 Maiores Pagamentos por Município e Credor)")

teste_join = duckdb.sql(f"""
    SELECT 
        ug.municipio,
        c.nome_credor,
        f.ano_referencia,
        SUM(f.valor_pago) AS total_pago
    FROM '{PATH}/fato_empenhos.parquet' f
    LEFT JOIN '{PATH}/dim_credor.parquet' c ON f.cpf_cnpj = c.cpf_cnpj
    LEFT JOIN '{PATH}/dim_unidade_gestora.parquet' ug ON f.codigo_unidade_gestora = ug.codigo_unidade_gestora
    GROUP BY ALL
    ORDER BY total_pago DESC
    LIMIT 5
""").df()

print(teste_join.to_string(index=False))

print("\n==================================================")
print("             VERIFICAÇÃO CONCLUÍDA                ")
print("==================================================")

      INICIANDO VERIFICAÇÃO DAS TABELAS           

1. CHECAGEM DA TABELA FATO (fato_empenhos)
   • Total de Linhas: 10,753,139
   • Total Empenhado: R$ 97,440,343,481.59
   • Total Liquidado: R$ 95,112,976,873.05
   • Total Pago:      R$ 92,691,346,666.02

2. CHECAGEM DE CHAVES PRIMÁRIAS DUPLICADAS NAS DIMENSÕES
   • dim_credor -> PK [cpf_cnpj]: OK (100% Única)
   • dim_unidade_gestora -> PK [codigo_unidade_gestora]: OK (100% Única)
   • dim_unidade_orcamentaria -> PK [codigo_unidade_orcamentaria]: OK (100% Única)
   • dim_funcao -> PK [codigo_funcao]: OK (100% Única)
   • dim_subfuncao -> PK [codigo_subfuncao]: OK (100% Única)
   • dim_programa -> PK [codigo_programa]: OK (100% Única)
   • dim_acao -> PK [codigo_acao]: OK (100% Única)
   • dim_categoria_economica -> PK [codigo_categoria_economica]: OK (100% Única)
   • dim_natureza -> PK [codigo_natureza]: OK (100% Única)
   • dim_modalidade_aplicacao -> PK [codigo_modalidade_aplicacao]: OK (100% Única)
   • dim_elemento_despesa -> P